# ComfyUI + LTX 2.5 | Google Colab G4

Clean setup for evaluating **LTX 2.5** with synchronized audio on a Colab G4 (Blackwell). The working installation, models, and outputs stay on Colab's **fast local disk** (`/content`), not mounted Google Drive.

**Run cells from top to bottom**: 1) Settings, 2) Install, 3) Models & workflows, 4) Launch. Open one of the downloaded workflows from **Workflows → Open** in ComfyUI (see folder printed in cell 3). No old Stable Diffusion models, alternative tunnels, or obsolete launch methods.

**Session persistence:** Colab local storage is temporary. Optional Google Drive **model archive** can prevent repeat internet downloads on subsequent sessions, but requires roughly **80+ GB of Drive quota** and copies the files back to fast local disk at startup. If Drive cannot hold the archive, downloads use local disk and must be repeated on a fresh runtime. Do not run inference from Drive. Saved videos can be copied to Drive separately.

**Security:** The temporary Cloudflare Quick Tunnel exposes ComfyUI publicly to anyone with its URL. Do not share the URL or add secrets to nodes. Stop the runtime when finished. Hugging Face may require accepting the LTX license and supplying a token for gated weights.

Official references: https://docs.ltx.io/open-source-model/integration-tools/comfy-ui and https://github.com/Lightricks/ComfyUI-LTXVideo/tree/master/example_workflows/2.5

In [ ]:
#@title 1. Settings (edit before running)
from pathlib import Path

WORKSPACE = Path("/content/ComfyUI")  # Always fast Colab VM disk
MODEL_REPO = "Lightricks/LTX-2.5"
USE_DRIVE_MODEL_ARCHIVE = False  #@param {type:"boolean"}
# Enable only if you have ~80+ GB free in Google Drive; startup stages files onto /content.
ARCHIVE_ROOT = "/content/drive/MyDrive/LTX25-Colab-Model-Archive"  #@param {type:"string"}
SAVE_OUTPUTS_TO_DRIVE = False  #@param {type:"boolean"}
# Output backup is a separate manual cell; generation always writes to /content/ComfyUI/output.
INSTALL_PROMPT_ENHANCER = True  #@param {type:"boolean"}
# Official workflows enable enhancement by default. Disable in individual workflows if omitted.

print("Local workspace:", WORKSPACE)
print("Persistent Drive model archive:", USE_DRIVE_MODEL_ARCHIVE)
print("Prompt enhancer:", INSTALL_PROMPT_ENHANCER)

In [ ]:
#@title 2. Install ComfyUI + LTX nodes on fast local disk
import os, sys, subprocess, shutil
from pathlib import Path

def run(*args, cwd=None):
    print("+", " ".join(map(str, args)), flush=True)
    subprocess.run([str(a) for a in args], cwd=str(cwd) if cwd else None, check=True)

def clone_or_pull(url, folder):
    folder = Path(folder)
    if (folder / ".git").exists():
        run("git", "-C", folder, "pull", "--ff-only")
    else:
        folder.parent.mkdir(parents=True, exist_ok=True)
        run("git", "clone", "--depth", "1", url, folder)

run("nvidia-smi")
print("Available VM disk space:", round(shutil.disk_usage("/content").free / 2**30, 1), "GiB")
if shutil.disk_usage("/content").free < 90 * 2**30:
    print("WARNING: <90 GiB free. BF16 models + downloads may need more disk; check Colab storage.")
run("apt-get", "update", "-qq")
run("apt-get", "install", "-y", "-qq", "ffmpeg")
run(sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "requests", "safetensors")
clone_or_pull("https://github.com/comfyanonymous/ComfyUI.git", WORKSPACE)
run(sys.executable, "-m", "pip", "install", "-q", "-r", WORKSPACE / "requirements.txt")
custom = WORKSPACE / "custom_nodes"
clone_or_pull("https://github.com/ltdrdata/ComfyUI-Manager.git", custom / "ComfyUI-Manager")
clone_or_pull("https://github.com/Lightricks/ComfyUI-LTXVideo.git", custom / "ComfyUI-LTXVideo")
ltx_req = custom / "ComfyUI-LTXVideo" / "requirements.txt"
if ltx_req.exists():
    run(sys.executable, "-m", "pip", "install", "-q", "-r", ltx_req)
else:
    print("LTX node pack does not expose requirements.txt; its latest README recommends uv sync for standalone setup.")
for folder in ("diffusion_models", "text_encoders", "vae", "latent_upscale_models", "loras"):
    (WORKSPACE / "models" / folder).mkdir(parents=True, exist_ok=True)
(WORKSPACE / "user" / "default" / "workflows").mkdir(parents=True, exist_ok=True)
run(sys.executable, "-c", "import torch; print('PyTorch:',torch.__version__,'CUDA:',torch.version.cuda,'GPU:',torch.cuda.get_device_name(0))")
print("ComfyUI installed locally:", WORKSPACE)

In [ ]:
#@title 3. Download required LTX 2.5 BF16 models + official ready-to-open workflows
import os, json, shutil, requests, getpass
from pathlib import Path
from huggingface_hub import hf_hub_download
from huggingface_hub.errors import HfHubHTTPError

# Compatible with official 2.5 two-stage, single-stage and audio-to-video graphs.
# Files from the same official HF subfolders are placed directly under ComfyUI/models.
MODEL_FILES = [
    ("Lightricks/LTX-2.5", "diffusion_models/ltx-2.5-22b-distilled-transformer-bf16.safetensors"),
    ("Lightricks/LTX-2.5", "text_encoders/gemma4-12b-with-proj-ltx-2.5-bf16.safetensors"),
    ("Lightricks/LTX-2.5", "vae/ltx-2.5-video-vae-bf16.safetensors"),
    ("Lightricks/LTX-2.5", "vae/ltx-2.5-video-vae-conv-bf16.safetensors"),
    ("Lightricks/LTX-2.5", "vae/ltx-2.5-audio-vae-bf16.safetensors"),
    ("Lightricks/LTX-2.5", "latent_upscale_models/ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors"),
]
if INSTALL_PROMPT_ENHANCER:
    MODEL_FILES.append(("Comfy-Org/gemma-4", "gemma4_e2b_it_bf16.safetensors"))

# Avoid storing live checkpoints on slow Drive. Archive is strictly copy-in/copy-out.
archive = None
if USE_DRIVE_MODEL_ARCHIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    archive = Path(ARCHIVE_ROOT)
    archive.mkdir(parents=True, exist_ok=True)
    free_gib = shutil.disk_usage(archive).free / 2**30
    print(f"Google Drive free space reported by mount: {free_gib:.1f} GiB")
    if free_gib < 80 and not any(archive.rglob("*.safetensors")):
        print("Drive archive disabled: initial full BF16 cache requires ~80+ GiB of free quota.")
        archive = None

# Hugging Face access may be gated. Prefer Colab Secrets key HF_TOKEN; never print it.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN") or None

for repo_id, rel in MODEL_FILES:
    relpath = Path(rel)
    folder = "text_encoders" if repo_id == "Comfy-Org/gemma-4" else relpath.parent.as_posix()
    filename = relpath.name
    target = WORKSPACE / "models" / folder / filename
    target.parent.mkdir(parents=True, exist_ok=True)
    stored = archive / folder / filename if archive else None
    if target.exists() and target.stat().st_size > 100_000_000:
        print("Already local:", filename)
        continue
    if stored and stored.exists() and stored.stat().st_size > 100_000_000:
        print("Staging from Drive archive to local SSD:", filename, flush=True)
        shutil.copy2(stored, target)
        continue
    print("Downloading:", repo_id, rel, flush=True)
    try:
        # Download into the final fast local model folder; HF resumes interrupted transfers.
        hf_hub_download(repo_id=repo_id, filename=(filename if repo_id == "Comfy-Org/gemma-4" else rel), local_dir=(target.parent if repo_id == "Comfy-Org/gemma-4" else WORKSPACE / "models"), token=HF_TOKEN)
    except HfHubHTTPError as exc:
        if getattr(exc.response, "status_code", None) in (401, 403):
            raise RuntimeError("Hugging Face access denied. Accept the model license on HF and set HF_TOKEN in Colab Secrets.") from exc
        raise
    assert target.exists() and target.stat().st_size > 100_000_000, f"Missing model after download: {target}"
    if stored:
        stored.parent.mkdir(parents=True, exist_ok=True)
        if shutil.disk_usage(stored.parent).free > target.stat().st_size + (2 * 2**30):
            print("Archiving to Drive (not used during generation):", filename, flush=True)
            tmp = stored.with_suffix(stored.suffix + ".partial")
            shutil.copy2(target, tmp)
            os.replace(tmp, stored)
        else:
            print("Drive quota insufficient to archive", filename, "— kept only on local runtime.")

OFFICIAL = "https://raw.githubusercontent.com/Lightricks/ComfyUI-LTXVideo/master/example_workflows/2.5/"
WORKFLOWS = [
    "LTX-2.5_T2V_I2V_Two_Stage_Distilled.json",
    "LTX-2.5_T2V_I2V_Single_Stage_Distilled.json",
    "LTX-2.5_A2V_Two_Stage_Distilled.json",
]
wf_dir = WORKSPACE / "user" / "default" / "workflows"
for filename in WORKFLOWS:
    response = requests.get(OFFICIAL + filename, timeout=120)
    response.raise_for_status()
    json.loads(response.text)
    (wf_dir / filename).write_text(response.text)
    print("Workflow ready:", filename)
print("\nAll essential model files are staged on FAST local disk.")
print("Workflows folder:", wf_dir)
print("Choose two-stage T2V/I2V for quality, single-stage for quick tests, A2V for existing dialogue.")
print("Note: if using INSTALL_PROMPT_ENHANCER=False, turn off prompt enhancement in the workflow.")

In [ ]:
#@title 4. Start ComfyUI using ONE Cloudflare Quick Tunnel
import os, re, time, subprocess, threading, urllib.request
from pathlib import Path

# Only create a tunnel for the current VM; never expose this URL publicly.
if subprocess.run(["bash","-lc","command -v cloudflared"],capture_output=True).returncode:
    deb = Path("/content/cloudflared.deb")
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb", deb)
    subprocess.run(["dpkg","-i",str(deb)],check=True)

# Restart-safe within the SAME runtime, but do not run this cell in parallel.
for key in ("COMFY_PROCESS", "CLOUDFLARED_PROCESS"):
    process = globals().get(key)
    if process is not None and process.poll() is None:
        process.terminate()
        try: process.wait(timeout=8)
        except subprocess.TimeoutExpired: process.kill()
time.sleep(2)

log = open("/content/comfyui.log","a",buffering=1)
COMFY_PROCESS = subprocess.Popen(
    ["python","main.py","--listen","127.0.0.1","--port","8188"],
    cwd=str(WORKSPACE),stdout=log,stderr=subprocess.STDOUT)
print("ComfyUI PID:", COMFY_PROCESS.pid)

for _ in range(150):
    if COMFY_PROCESS.poll() is not None:
        raise RuntimeError("ComfyUI exited. Inspect /content/comfyui.log (see diagnostics cell).")
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats",timeout=2).read()
        break
    except Exception: time.sleep(2)
else:
    raise TimeoutError("ComfyUI did not become ready; inspect /content/comfyui.log")

tunnel_log = open("/content/cloudflared.log","a",buffering=1)
CLOUDFLARED_PROCESS = subprocess.Popen(
    ["cloudflared","tunnel","--no-autoupdate","--url","http://127.0.0.1:8188"],
    stdout=tunnel_log,stderr=subprocess.STDOUT)
url = None
for _ in range(90):
    text = Path("/content/cloudflared.log").read_text(errors="replace")
    matches = re.findall(r"https://[\w-]+\.trycloudflare\.com", text)
    if matches: url = matches[-1]; break
    if CLOUDFLARED_PROCESS.poll() is not None: break
    time.sleep(2)
if not url:
    raise RuntimeError("Tunnel did not return a URL. Inspect /content/cloudflared.log.")
print("\nOPEN COMFYUI:", url)
print("Use Workflows → Open and select LTX-2.5_T2V_I2V_Two_Stage_Distilled.json")
print("Keep Colab runtime connected while generating. Stop runtime to stop billing.")
print("Cloudflare Quick Tunnel is not password protected. Keep this URL private.")

In [ ]:
#@title 5. Diagnostics (run only when something fails)
from pathlib import Path
import shutil, subprocess
print("Free local disk GiB:",round(shutil.disk_usage("/content").free / 2**30,1))
subprocess.run(["nvidia-smi"])
for name in ("comfyui.log","cloudflared.log"):
    p=Path("/content")/name
    if p.exists():
        print("\n---",name,"(last 70 lines) ---")
        print("\n".join(p.read_text(errors="replace").splitlines()[-70:]))

In [ ]:
#@title 6. Optional: back up finished videos to Google Drive (not used during inference)
from pathlib import Path
import shutil
if not SAVE_OUTPUTS_TO_DRIVE:
    print("Output backup disabled. Videos are in", WORKSPACE / "output")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    src = WORKSPACE / "output"
    dst = Path("/content/drive/MyDrive/LTX25-Colab-Outputs")
    dst.mkdir(parents=True,exist_ok=True)
    copied=0
    for f in src.rglob("*"):
        if f.is_file():
            target=dst / f.relative_to(src)
            target.parent.mkdir(parents=True,exist_ok=True)
            if not target.exists() or f.stat().st_size!=target.stat().st_size:
                shutil.copy2(f,target)
                copied+=1
    print("New/changed files backed up:", copied, "Destination:",dst)